In [1]:
# get a file from google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install torchview

In [3]:
import matplotlib.pyplot as plt

import numpy as np

import matplotlib.pyplot as plt

import torch
from scipy.stats import spearmanr

import joblib
import numpy as np
import torch
import abc
import os
import copy
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as f

# Metric Learning
## High-level Description
Standard reinforcement learning has many issues: it can be unstable (requiring multiple seeds) and sample inefficient (needing costly environment interactions). To address this, recent methods treat RL as a supervised learning problem—mapping states and desired goals to optimal actions. These outcome-conditional behavioral cloning (OCBC) algorithms achieve strong results on common benchmarks.

In this homework, you will implement two simplified OCBC methods:

1. One predicts the temporal distance between two states.
2. The other uses contrastive learning.

You will compare them using various metrics, test their stitching capabilities, and present your findings in a report.

## Points
You will be able to get 10 points in total for the homework, it will be split as follows:
- Implementing various "metrics" [1pt]:
    - Correlation and distance plots [0.25pt]
    - Heatmaps [0.25pt]
    - Solved Rate [0.5pt]
- Implement the network architecture, based on BRO net [0.5pt]
- Implement contrastive training correctly [2pt]
- Implementing the supervised training correctly [1.5pt]
- Based on the metrics you implemented, describe how well both implemented methods are working. [1pt]
- Compare the supervised and contrastive methods performance, how do they compare in terms of sample efficiency? [1pt]
- Compare the stitching abilities of the supervised and contrastive method. [1pt]
- Report quality [2pt] (multiplied by the sum of the rest and divided by max of the rest)



## Report
Your solution will be graded based both on the code and the report. A solution without a report gets 0 points. Your report should be a pdf file, based on the provided latex template.

## Datasets

You have been given two [dataset](https://drive.google.com/drive/folders/1-YKwZpttn9Of5zR_poT-lO5coMZ0tvmN?usp=sharing). You can access them, if you log into your university gmail account. If you are using Colab, it's best to put them on Google Drive associated with your University Google account (it's also a good idea to save your checkpoints there). A single 20 by 20 maze has been chosen for both of them.

Each dataset consists of two files, file with trajectories and file with lengths of the corresponding trajectories. They are compressed using the `joblib` library.

The normal dataset contains optimal trajectories going from a randomly sampled start point in the maze to a randomly sampled goal state in the maze.

The 'stitching' dataset contains a subset of trajectories, that have the goal and start states within the same quarter of the maze board.


*   The normal dataset:
    * `maze_trajectories.pkl`
    * `maze_lens.pkl`
*   The stitching dataset:
    * `stitching_trajectories.pkl`
    * `stitching_lens.pkl`

In the saved trajectories, the walls are denoted by 1, start state by 2 and goal state by 3. The are ordered from last to first state. In the last state, the position of the goal state and start state is the same.

The implemented `AbsDataset` reads the trajectories and allows for getting a batch of trajectories from them.
You are to implement two classes deriving from `AbsDataset` - `ContrastiveDataset` and `SupervisedDataset`.

### Supervised Dataset
In Supervised Dataset, the method `get_batch` should return a batch of pairs of states from the same trajectory, they should both be sampled uniformly, and the first state should be earlier in trajectory. For each batch element, a label should be returned, the label being the distance between the sampled states.

### Contrastive Dataset
In the Contrastive Dataset, the method `get_batch` should return a batch of pairs of states from the same trajectory. The first element from the pair should be sampled randomly from the trajectory, while the second should be sampled from an exponential distribution, as described [here](https://arxiv.org/pdf/2206.07568). So, if the first element sampled has index $i$, the probability of the second one having index $i + j$ should be proportional to $\gamma^j$.




In [ ]:
traj_path = '/content/drive/MyDrive/HW_2_Trajectories/maze_trajectories.pkl'
len_path =  '/content/drive/MyDrive/HW_2_Trajectories/maze_lens.pkl'

### DO NOT EDIT ###
class AbsDataset(abc.ABC):
    def __init__(self, path, lens_path, n_train_trajectories, gamma=None):
        # super(Dataset).__init__()
        self.trajectories = torch.from_numpy(joblib.load(path))
        self.lens = torch.from_numpy(joblib.load(lens_path))

        # Only take the non-empty trajectories from the files
        mask = self.lens == 0

        assert (torch.logical_or(self.trajectories==2, self.trajectories==2)).sum() == sum(self.lens - 1), \
        f'{(torch.logical_or(self.trajectories==2, self.trajectories==3)).sum()}, {sum(self.lens)}'

        self.trajectories = self.trajectories[~mask]
        self.lens = self.lens[~mask]

        assert (torch.logical_or(self.trajectories==2, self.trajectories==2)).sum() == sum(self.lens - 1), \
        f'{(torch.logical_or(self.trajectories==2, self.trajectories==3)).sum()}, {sum(self.lens)}'

        permutation = np.random.permutation(len(self.trajectories))


        self.train_trajectories = self.trajectories[permutation][:n_train_trajectories]
        self.train_lens = self.lens[permutation][:n_train_trajectories]

        assert (torch.logical_or(self.train_trajectories==2, self.train_trajectories==2)).sum() == sum(self.train_lens - 1), \
        f'{(torch.logical_or(self.train_trajectories==2, self.train_trajectories==3)).sum()}, {sum(self.lens)}'

        self.test_trajectories = self.trajectories[permutation][n_train_trajectories:]
        self.test_lens = self.lens[permutation][n_train_trajectories:]

        assert (self.test_trajectories==2).sum() == sum(self.test_lens - 1)

        self.ind = 0
        self.gamma = gamma
        self.max_horizon = 400
        self.device = "cuda" if torch.cuda.is_available() else "cpu"


    def _get_trajs(self, n_traj, split):
        if split not in ['train', 'test']:
            raise ValueError()

        if split == 'train':
            trajectories = self.train_trajectories
            lens = self.train_lens
        else:
            trajectories = self.test_trajectories
            lens = self.test_lens

        if n_traj > len(trajectories):
            n_traj = len(trajectories)

        if self.ind + n_traj > len(trajectories):
            split_1 = trajectories[self.ind:]
            split_2 = trajectories[:self.ind + n_traj - len(trajectories)]
            current_trajectories = torch.concatenate([split_1, split_2], axis=0)

            split_1 = lens[self.ind:]
            split_2 = lens[:self.ind + n_traj - len(trajectories)]
            current_lens = torch.concatenate([split_1, split_2], axis=0)
            permutation = np.random.permutation(len(trajectories))

            if split == 'train':
                self.train_trajectories = self.train_trajectories[permutation]
                self.train_lens = self.train_lens[permutation]
            else:
                self.test_trajectories = self.test_trajectories[permutation]
                self.test_lens = self.test_lens[permutation]

            self.ind = 0

        else:
          current_trajectories = trajectories[self.ind:self.ind + n_traj]
          current_lens = lens[self.ind:self.ind + n_traj]

        # The generated maze trajectories have states of a form where 2 is the start and 3 is the end
        # The first element of each trajectory is the solved state
        # Here we replace all 3 with 2
        # Since this causes the training to be easier, as we only calculate the distance between
        # Two fields with value 2, not taking into account where the goal is
        mask1 = torch.zeros_like(current_trajectories).to(self.device)
        mask1[torch.arange(len(current_trajectories)), 0] = 1
        mask2 = (current_trajectories == 3).to(self.device)
        mask3 = current_lens < len(current_trajectories[0])
        mask3 = mask3.unsqueeze(1).unsqueeze(2).unsqueeze(3).repeat(1, len(current_trajectories[0]), 1, 1).to(self.device)

        mask = mask1 * mask2 * mask3

        current_trajectories[mask.to(bool)] = 2
        current_trajectories[current_trajectories == 3] = 0

        assert (current_trajectories == 3).sum() == 0
        assert ((current_trajectories == 2).sum(dim=(1, 2, 3)) == current_lens).all(), f'{(current_trajectories[0] == 2).sum()} {current_lens[0]}'
        assert ((current_trajectories == 2).sum(axis = (2, 3)) <= 1).all()

        self.ind += n_traj
        self.ind %= len(self.trajectories)

        return current_trajectories, current_lens

    @abc.abstractmethod
    def get_batch(self, batch_size):
        pass

    def get_trajectory(self):
        traj, len = self._get_trajs(1)
        traj = traj.squeeze()
        traj = traj.flatten(1)
        len = len.item()

        traj = traj[:len]

        return traj.to(self.device).to(torch.float32)
### DO NOT EDIT ###

class ContrastiveDataset(AbsDataset):
    def get_batch(self, batch_size, split):
        ### TODO ###
        pass
        ### TODO ###

class SupervisedDataset(AbsDataset):
    def get_batch(self, batch_size, split):
        ### TODO ###
        pass
        ### TODO ###

class Dataloader():
    def __init__(self, dataset, batch_size, split='train'):
        self.dataset = dataset
        self.batch_size = batch_size
        self.split = split

    def __iter__(self):
        return self

    def __next__(self):
        return self.dataset.get_batch(self.batch_size, split=self.split)

## BRO Net
Implement the architecture based on the [BRO paper](https://arxiv.org/pdf/2405.16158).

In [ ]:
class BroNet(nn.Module):
    ### TODO ####
    pass
    ### TODO ###

In [ ]:
### DO NOT EDIT ###
from torchview import draw_graph
model = BroNet()
viz = draw_graph(model, input_size=(1, 400), expand_nested=True, device='meta')
viz.visual_graph
### DO NOT EDIT ###

RuntimeError: Failed to run torchgraph see error message

## Contrastive Loss
Implement the symmetrized version of the [InfoNCE](https://arxiv.org/pdf/1807.03748) loss. You don't need to read this paper, but you can scan the intro for some context.

Given two batches of corresponding embeddings $\{z_i\}_{i=1}^N$ and $\{z_i'\}_{i=1}^N$, the **InfoNCE loss** for predicting $z_i'$ from $z_i$ is:

$$
\mathcal{L}_{\text{InfoNCE}}(z_i, z_i') = -\log \frac{\exp(\text{sim}(z_i, z_i') / \tau)}{\sum_{j=1}^{N} \exp(\text{sim}(z_i, z_j') / \tau)}
$$

Where:

* $\text{sim}(a, b) = -||a - b||_2$ is the $\ell_2$ norm,
* $\tau$ is a temperature parameter (a good default value for $\tau$ is the square root of representation dimension),
* The denominator includes the positive and all negatives from the batch.

---

### Symmetrized InfoNCE Loss

To symmetrize, we apply the InfoNCE loss in both directions:

* Predict $z_i'$ from $z_i$
* Predict $z_i$ from $z_i'$

The **Symmetrized InfoNCE** is then:

$$
\mathcal{L}_{\text{Symm-InfoNCE}} = \sum_{i=1}^{N} \left[ \mathcal{L}_{\text{InfoNCE}}(z_i, z_i') + \mathcal{L}_{\text{InfoNCE}}(z_i', z_i) \right]
$$

This encourages mutual information maximization in both directions.

In [ ]:
def contrastive_loss(psi_0, psi_T):
        ### TODO ###
        pass
        ### TODO ###
        return loss.mean()


## Training
### Supervised Network Training
The supervised network should receive as input a concatenated pair of state and predict the distance between them. It should be trained with a classification-based objective.

### Contrastive Network Training
The contrastive network should receive as input a single state and return its representation.

In [ ]:
class Trainer(abc.ABC):
    def __init__(self, checkpoint_frequency, output_dir, train_steps, traj_path, model, n_train_trajectories, lr, batch_size):
        self.model = model
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device)


        self.train_steps = train_steps
        self.output_dir = output_dir
        self.checkpoint_frequency = checkpoint_frequency

        self.lr = lr
        self.batch_size = batch_size
        self.losses = []

        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        self.init_dataset(traj_path, n_train_trajectories)

    @abc.abstractmethod
    def calculate_loss(self, data):
        pass

    @abc.abstractmethod
    def init_dataset(self, traj_path, n_train_trajectories):
        pass

    def train_model(self):
        ### TODO ###
        pass
        ### TODO ###

class ContrastiveTrainer(Trainer):
    def __init__(self, **kwargs):
        super(ContrastiveTrainer, self).__init__(**kwargs)

    ### TODO ###
    pass
    ### TODO ###

class SupervisedTrainer(Trainer):
    def __init__(self, **kwargs):
        super(SupervisedTrainer, self).__init__(**kwargs)
        self.loss_fn = torch.nn.CrossEntropyLoss()

    ### TODO ###
    pass
    ### TODO ###



## Value Estimators
The goal of this class is to create an abstraction for predicting distance between the two state. Implement one for supervised network and one for the contrastive network.

In [ ]:
class ValueEstimator(abc.ABC):
    def __init__(self, model):
        self.model = model

    def get_solved_distance(self, state_str: np.array, goal: np.array):
        return self.get_solved_distance_batch(np.expand_dims(state_str, 0), np.expand_dims(goal, 0))

    @abc.abstractmethod
    def calculate_distance(self, num_state, num_goal):
        pass

    def get_solved_distance_batch(self, state_str: np.array, goal: np.array):
        assert (state_str == 2).sum() == len(state_str)
        assert (state_str == 3).sum() == 0

        assert (goal == 2).sum() == len(state_str)
        assert (goal == 3).sum() == 0

        num_state = torch.tensor(state_str).to(torch.float32)
        num_goal = torch.tensor(goal).to(torch.float32)
        self.model.eval()

        return self.calculate_distance(num_goal, num_state)

class ValueEstimatorContrastive(ValueEstimator):
    def calculate_distance(self, num_state, num_goal):
        ### TODO ###
        pass
        ### TODO ###


class ValueEstimatorSupervised(ValueEstimator):
    def calculate_distance(self, num_state, num_goal):
        ### TODO ###
        pass
        ### TODO ###


In [ ]:
def build_board(board, end, start):
    wall_pattern = copy.deepcopy(board[board < 2]).reshape((20, 20))

    if wall_pattern[start] != 0:
        raise ValueError()

    if wall_pattern[end] != 0:
        raise ValueError()

    wall_pattern[start] = 2
    wall_pattern[end] = 3

    return torch.from_numpy(wall_pattern)

In [ ]:
class Maze():
  def __init__(self, wall_pattern):
    i, j = torch.randint(0, 20, (1,)), torch.randint(0, 20, (1,))
    while wall_pattern[i, j] != 0:
        i, j = torch.randint(0, 20, (1,)), torch.randint(0, 20, (1,))

    self.solved_state = copy.deepcopy(wall_pattern)
    self.solved_state[i, j] = 2

    i, j = torch.randint(0, 20, (1,)), torch.randint(0, 20, (1,))
    while wall_pattern[i, j] != 0:
        i, j = torch.randint(0, 20, (1,)), torch.randint(0, 20, (1,))

    self.current_state = copy.deepcopy(wall_pattern)
    self.current_state[i, j] = 2

  def solved(self):
    return (self.solved_state == self.current_state).all()

  def get_current_state(self):
    return self.current_state

  def get_solved_state(self):
    return self.solved_state

  def do_step(self, action):
    assert action in [0, 1, 2, 3]
    pos = torch.where(self.current_state == 2)
    i, j = pos[0][0], pos[1][0]
    if action == 0:
      if i > 0 and self.current_state[i-1][j] != 1:
        self.current_state[i][j] = 0
        self.current_state[i-1][j] = 2
    elif action == 1:
      if i < 19 and self.current_state[i+1][j] != 1:
        self.current_state[i][j] = 0
        self.current_state[i+1][j] = 2
    elif action == 2:
      if j > 0 and self.current_state[i][j-1] != 1:
        self.current_state[i][j] = 0
        self.current_state[i][j-1] = 2
    elif action == 3:
      if j < 19 and self.current_state[i][j+1] != 1:
        self.current_state[i][j] = 0
        self.current_state[i][j+1] = 2

    return self.current_state, self.solved()

  def try_step(self, action, temp_state):
    assert action in [0, 1, 2, 3]
    pos = torch.where(temp_state == 2)
    i, j = pos[0][0], pos[1][0]
    if action == 0:
      if i > 0 and temp_state[i-1][j] != 1:
        temp_state[i][j] = 0
        temp_state[i-1][j] = 2
    elif action == 1:
      if i < 19 and temp_state[i+1][j] != 1:
        temp_state[i][j] = 0
        temp_state[i+1][j] = 2
    elif action == 2:
      if j > 0 and temp_state[i][j-1] != 1:
        temp_state[i][j] = 0
        temp_state[i][j-1] = 2
    elif action == 3:
      if j < 19 and temp_state[i][j+1] != 1:
        temp_state[i][j] = 0
        temp_state[i][j+1] = 2

    return temp_state, (temp_state == self.solved_state).all()

  def render(self):
    state = copy.deepcopy(self.current_state)
    state[torch.where(self.solved_state == 2)] = 3
    plt.imshow(state, cmap='viridis', vmin=-0.5, vmax=state.max())
    plt.colorbar()
    plt.gca().set_aspect('equal')
    plt.show()

  def render_state(self, state):
    state = copy.deepcopy(state)
    state[torch.where(self.solved_state == 2)] = 3
    plt.imshow(state, cmap='viridis', vmin=-0.5, vmax=state.max())
    plt.colorbar()
    plt.gca().set_aspect('equal')
    plt.show()




## Metrics

### Plot Heatmap
For a given maze board, plot the distance to the goal state from each possible starting position.

### Correlation and Trajetory Distances
Calculate the average Spearman Rank correlation between the predicted distance and the position in the trajectory. For a given number of trajectories, plot how the predicted distance changes compared to the distance until the trajectory end.

### Solved Rate
Use the learned networks as distance function for solving the maze, by using the [BestFS](https://en.wikipedia.org/wiki/Best-first_search) algorithm.

In [ ]:
import copy
def plot_maze_heatmap(init_state, value_estimator):
  ### TODO ###
    pass
  ### TODO ###

dataset = SupervisedDataset(traj_path, len_path, 4000, 0.9)
test_trajectories, test_lengths = dataset._get_trajs(10, split='test')


In [ ]:
a = joblib.load(traj_path)

board = a[0][0]
board[board > 1] = 0
env = Maze(torch.from_numpy(board))

In [ ]:
def get_correlation(trajectories, lengths, value_estimator):
    correlations = []
    for trajectory, length in zip(trajectories, lengths):
        goal = np.expand_dims(trajectory[0], 0)
        l = length.to(int).item()
        distances = value_estimator.get_solved_distance_batch(trajectory[:l].reshape((l, -1)), np.tile(goal, ((l,1, 1))).reshape((l, -1)))
        correlation = spearmanr(distances.cpu(), np.arange(len(distances.cpu()))).statistic
        if not np.isnan(correlation):
            correlations.append(correlation)

    return sum(correlations)/len(correlations)


def plot_distances(trajectories, lengths, value_estimator):
    ### TODO ###
    pass
    ### TODO ###



In [ ]:

import torch

def check_solved_rate(value_estimator, n_instances, do_search=True):
    ### TODO ###

    for i in range(n_instances):
        # Create a set of seen states
        # Create the maze environment

        # Define a priority queue

        # Count how many states have been seen

        while not env.solved() and len(queue) > 0:
            # Get the solved state

            # Get the next best state

            # Iterate through the state's neighbours
            # and add them to the priority queue if they have not been seen
            # If do_search is False, add only the best state

    # What fraction of the states has been solved
    print("Solved rate:", solved_cnt/n_instances)

    # How many states on average need to be seen to solve a problem instance
    print("Average expanded:", sum(expanded_nodes) / len(expanded_nodes))
    ### TODO ###

## Stitching in Behavior Cloning

**Stitching** refers to the idea of combining segments from different trajectories to form a new, coherent trajectory that the agent has not explicitly observed. If two trajectory fragments align at a common or similar state, they can be "stitched" together to create a longer path that may lead to higher rewards or better behavior.

In **behavior cloning**, where the agent learns to mimic actions from demonstration data, stitching becomes a key challenge. If successful behavior is spread across multiple partial trajectories, the agent may never see a complete example of optimal behavior. Since behavior cloning learns purely from observed sequences without reasoning about outcomes, it cannot infer that stitching parts of different demonstrations could result in better performance. This limits generalization, especially when the dataset is sparse or suboptimal.
